# Stage 2: Converting to the OpenAI Agents SDK
### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

---

In Stage 1 we built the recipe assistant as a raw `client.chat.completions.create()` call. That's the simplest possible thing — a system prompt, a user message, and a response.

Before we add tools, retrieval, or any agentic behavior, let's first move to a proper **agent framework**. The **OpenAI Agents SDK** (`openai-agents`) gives us a declarative `Agent` object that will make it much easier to add capabilities later.

This notebook is a focused conversion. We take the exact same system prompt and behavior from Stage 1, express it as an `Agent`, and confirm everything still works — same tracing, same evaluation.

By the end of this notebook you will understand:

- What the `Agent` object is and what it replaces
- How to point the SDK at Gemini (or any OpenAI-compatible provider)
- How `Runner.run_sync()` replaces `client.chat.completions.create()`
- How `mlflow.openai.autolog()` traces SDK calls automatically
- How to write a `predict_fn` wrapper so the agent works with `mlflow.genai.evaluate()`

> **Prerequisites:** Complete the Stage 1 eval fundamentals notebook or be familiar with the recipe assistant system prompt and `predict_fn` contract.

---
## 0 — Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

In [ ]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("recipe-agent-2")
mlflow.openai.autolog()

---
## 1 — Where We Left Off

Here's the recipe assistant from Stage 1 — a raw completions call wrapped in a `predict_fn`:

```python
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

SYSTEM_PROMPT = """You are a helpful cooking assistant for Mise en Place..."""

def recipe_agent(inputs: dict) -> str:
    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite-preview",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": inputs["question"]},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content
```

This works, but it's just a function call. There's no structure for adding tools, guardrails, or multi-turn conversation. The `Agent` object gives us a place to attach all of that.

---
## 2 — What Is the OpenAI Agents SDK?

The `openai-agents` package is a lightweight framework for building agents. At its core, it provides:

| Concept | What it does |
|---|---|
| **`Agent`** | A declarative object that holds instructions, model config, and tools |
| **`Runner`** | Executes the agent — handles the LLM call, tool dispatch loop, and conversation state |
| **`@function_tool`** | Turns a Python function into a tool the agent can call (we'll use this in the next notebook) |

The SDK is **provider-agnostic** — it defaults to OpenAI but works with any OpenAI-compatible API via `OpenAIChatCompletionsModel`. This is how we'll keep using Gemini.

### What changes vs raw completions?

| Raw completions | Agents SDK |
|---|---|
| You manage the `messages` list | The `Runner` manages conversation state |
| You call `client.chat.completions.create()` directly | `Runner.run_sync()` calls the model for you |
| You parse `response.choices[0].message.content` | `result.final_output` gives you the text |
| Adding tools means writing a dispatch loop | Adding tools means passing them to the `Agent` |
| Tracing needs `@mlflow.trace` | `mlflow.openai.autolog()` traces everything automatically |

---
## 3 — Configuring the Model Provider

The Agents SDK defaults to calling OpenAI's API. Since we're using Gemini, we need to create a custom model that points the SDK at Google's OpenAI-compatible endpoint.

This is the same `base_url` and `api_key` pattern from Stage 1, just wrapped in the SDK's model object.

In [ ]:
from agents import AsyncOpenAI, OpenAIChatCompletionsModel

# The SDK uses an async client internally
gemini_client = AsyncOpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# Wrap it in the SDK's model object
model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite-preview",
    openai_client=gemini_client,
)

print("Model configured: gemini-3.1-flash-lite-preview via OpenAI-compatible endpoint")

### Why `AsyncOpenAI`?

The Agents SDK is async internally — `Runner.run()` is an `async` function. Even when we use `Runner.run_sync()` (the synchronous wrapper), it runs an async event loop under the hood. The `AsyncOpenAI` client is what the SDK expects.

### Other providers

This same pattern works for any OpenAI-compatible endpoint:

```python
# Ollama (local)
client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="unused")
model = OpenAIChatCompletionsModel(model="llama3", openai_client=client)

# Azure OpenAI
client = AsyncOpenAI(base_url="https://your-resource.openai.azure.com/", api_key=os.environ["AZURE_KEY"])
model = OpenAIChatCompletionsModel(model="gpt-4o", openai_client=client)

# OpenAI (default — you don't even need this, it's the default)
model = "gpt-4o-mini"
```

---
## 4 — Creating the Agent

Now we create the `Agent` object. For this notebook it's simple — just a name, instructions (our system prompt), and the model. No tools yet.

In [ ]:
from agents import Agent

SYSTEM_PROMPT = """You are a helpful cooking assistant for Mise en Place, a cooking app.
Answer questions about recipes, cooking techniques, ingredients, and meal planning.

Guidelines:
- Always include specific measurements and time estimates
- Be conversational and encouraging, like a knowledgeable friend
- When giving recipes, briefly mention a vegetarian or dietary alternative if one exists
- For food safety questions, be accurate and include appropriate caveats"""

recipe_agent = Agent(
    name="Mise en Place Assistant",
    instructions=SYSTEM_PROMPT,
    model=model,
)

print(f"Agent created: {recipe_agent.name}")
print(f"Model: gemini-3.1-flash-lite-preview")
print(f"Tools: {recipe_agent.tools}")

That's the entire conversion. Compare this to the raw completions version:

| Before (raw) | After (SDK) |
|---|---|
| `client = OpenAI(api_key=..., base_url=...)` | `gemini_client = AsyncOpenAI(api_key=..., base_url=...)` |
| `SYSTEM_PROMPT = "..."` | `SYSTEM_PROMPT = "..."` (same) |
| `client.chat.completions.create(model=..., messages=[...])` | `Agent(name=..., instructions=..., model=...)` |

The system prompt is identical. The model is the same. We've just moved from an imperative API call to a declarative agent definition.

---
## 5 — Running the Agent

The `Runner` class executes the agent. It has three methods:

| Method | When to use |
|---|---|
| `Runner.run_sync(agent, input)` | Notebooks, scripts, synchronous code |
| `await Runner.run(agent, input)` | Async code, web servers |
| `Runner.run_streamed(agent, input)` | When you want streaming output |

All three return a `RunResult` object. The text response is in `.final_output`.

In [ ]:
from agents import Runner

result = Runner.run_sync(recipe_agent, "How do I make scrambled eggs?")
print(result.final_output)

In [ ]:
result = Runner.run_sync(recipe_agent, "How do I know when chicken breast is safely cooked?")
print(result.final_output)

In [ ]:
result = Runner.run_sync(recipe_agent, "What can I use instead of eggs in baking?")
print(result.final_output)

### What's in a `RunResult`?

The `RunResult` object contains more than just the text output. Let's inspect it.

In [ ]:
result = Runner.run_sync(recipe_agent, "How do I caramelize onions?")

print(f"final_output type: {type(result.final_output)}")
print(f"final_output length: {len(result.final_output)} chars")
print(f"last_agent: {result.last_agent.name}")
print(f"new_items: {len(result.new_items)} items")
print()
print("Items:")
for item in result.new_items:
    print(f"  - {type(item).__name__}")

---
## 6 — Automatic Tracing

In Stage 1, we used `mlflow.openai.autolog()` to trace `client.chat.completions.create()` calls. The same line traces the Agents SDK — no additional setup needed.

Open the MLflow UI and look at the `recipe-assistant-agents-sdk` experiment. You should see traces for each `Runner.run_sync()` call above.

The trace structure for a simple agent (no tools) looks like:

```
Agent run (Mise en Place Assistant)      ← root span
  └── chat.completions.create            ← the LLM call
```

Once we add tools in the next notebook, you'll see tool spans nested inside the agent span.

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

---
## 7 — The `predict_fn` Adapter

Our evaluation pipeline from Stage 1 expects a function with this signature:

```python
def predict_fn(inputs: dict) -> str
```

The Agents SDK has a different interface: `Runner.run_sync(agent, string) -> RunResult`. We need a thin adapter to bridge the two.

This is the same pattern we used in Stage 1 — the only difference is what's inside the function.

In [ ]:
def predict_fn(inputs: dict) -> str:
    """Adapter: mlflow.genai.evaluate() → OpenAI Agents SDK."""
    result = Runner.run_sync(recipe_agent, inputs["question"])
    return result.final_output


# Sanity check — confirm it matches the contract
output = predict_fn({"question": "How do I make guacamole?"})
print(f"Return type: {type(output).__name__}")  # str
print(f"Length: {len(output)} chars")
print(output[:200], "...")

Two lines. The adapter:
1. Extracts `question` from the `inputs` dict (the eval dataset contract)
2. Passes it to `Runner.run_sync()` (the Agents SDK contract)
3. Returns `.final_output` as a string (the eval framework contract)

This pattern is the same regardless of what agent framework you use:

```python
# LangGraph
def predict_fn(inputs: dict) -> str:
    result = graph.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})
    return result["messages"][-1].content

# CrewAI
def predict_fn(inputs: dict) -> str:
    result = crew.kickoff(inputs={"question": inputs["question"]})
    return str(result)
```

The evaluation pipeline doesn't care what's behind `predict_fn`. This decoupling lets you swap frameworks without rewriting your eval suite.

---
## 8 — Running the Evaluation Pipeline

Let's confirm the SDK-based agent produces the same quality results as the raw completions version. We'll use the same eval dataset and scorers from Stage 1.

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Guidelines

eval_dataset = [
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            "expected_facts": ["avocado", "lime", "salt", "onion", "cilantro"],
        },
    },
    {
        "inputs": {"question": "What can I use instead of eggs in baking?"},
        "expectations": {
            "expected_facts": ["applesauce", "banana", "flax"],
        },
    },
    {
        "inputs": {"question": "How do I know when chicken breast is safely cooked?"},
        "expectations": {
            "expected_facts": ["165", "internal temperature", "thermometer"],
        },
    },
    {
        "inputs": {"question": "How do I make pan gravy from drippings?"},
        "expectations": {
            "expected_response": (
                "After cooking meat, keep the drippings and browned bits in the pan. "
                "Place over medium heat, sprinkle in flour and stir to combine with the fat. "
                "Gradually add stock or broth while stirring, scraping up the flavorful bits "
                "from the bottom. Simmer for a few minutes until thickened to your liking."
            ),
        },
    },
    {
        "inputs": {"question": "How do I caramelize onions?"},
        "expectations": {
            "expected_facts": ["low heat", "30", "butter"],
        },
    },
    {
        "inputs": {"question": "How long does it take to make homemade pizza dough?"},
        "expectations": {
            "expected_facts": ["rise", "hour"],
        },
    },
]

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="uses_specific_measurements",
            guidelines=(
                "When providing a recipe or cooking instructions, the response must include "
                "specific measurements rather than vague quantities like 'some' or 'a handful'."
            ),
        ),
        Guidelines(
            name="includes_time_estimate",
            guidelines=(
                "Responses about recipes or cooking processes must mention how long the process takes."
            ),
        ),
    ],
)

print("=== Agents SDK evaluation ===")
print(results.metrics)

In [ ]:
results.tables["eval_results_table"]

### Comparing to Stage 1

The scores should be very similar to Stage 1's evaluation runs. We changed the *plumbing* (how the LLM is called), not the *behavior* (what it's asked to do). Any small differences come from LLM non-determinism, not from the SDK.

This confirms that the conversion is safe — the SDK is a drop-in replacement for the raw completions call.

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| `AsyncOpenAI` | Created an async client pointing at Gemini's OpenAI-compatible endpoint |
| `OpenAIChatCompletionsModel` | Wrapped the client so the Agents SDK can use it |
| `Agent(name, instructions, model)` | Declared the recipe assistant as an agent object |
| `Runner.run_sync()` | Executed the agent synchronously, replacing `client.chat.completions.create()` |
| `result.final_output` | Got the text response, replacing `response.choices[0].message.content` |
| `mlflow.openai.autolog()` | Traced all SDK calls automatically — same line as Stage 1 |
| `predict_fn` adapter | Two-line wrapper bridging the SDK to `mlflow.genai.evaluate()` |

### What changed, what didn't

| | Before (raw completions) | After (Agents SDK) |
|---|---|---|
| **System prompt** | Same | Same |
| **Model** | Same | Same |
| **How you call the LLM** | `client.chat.completions.create(...)` | `Runner.run_sync(agent, ...)` |
| **How you get the response** | `response.choices[0].message.content` | `result.final_output` |
| **How you trace** | `mlflow.openai.autolog()` | `mlflow.openai.autolog()` (same) |
| **How you evaluate** | `predict_fn(inputs) -> str` | `predict_fn(inputs) -> str` (same) |
| **Adding tools later** | Write a dispatch loop | Pass them to `Agent(tools=[...])` |

The `Agent` object doesn't do much yet — it's just a nicer way to call the same model. But it gives us a clean place to attach tools, guardrails, and handoffs in the next notebooks.

---

**Next →** We add tools to the agent using `@function_tool` and evaluate tool-calling quality with `ToolCallCorrectness` and `ToolCallEfficiency`.